In [ ]:
import numpy as np
import pandas as pd

admissions = pd.read_csv("binary.csv")

# Make dummy variables for rank
data = pd.get_dummies(admissions, prefix="rank", columns=["rank"])

# Standardize features
for field in ["gre", "gpa"]:
    mean, std = data[field].mean(), data[field].std()
    data.loc[:, field] = (data[field] - mean) / std

# Split off random 10% of the data for testing
np.random.seed(42)
sample = np.random.choice(data.index, size=int(len(data) * 0.9), replace=False)
data, test_data = data.loc[sample], data.drop(sample)

# Split into features and targets
features, targets = data.drop("admit", axis=1), data["admit"]
features_test, targets_test = test_data.drop("admit", axis=1), test_data["admit"]

In [ ]:
import numpy as np
from data_prep import features, targets, features_test, targets_test


def sigmoid(x):
    """
    Calculate sigmoid
    """
    return 1 / (1 + np.exp(-x))


def update_weights(weights, features, targets, learnrate):
    """
    Complete a single epoch of gradient descent and return updated weights
    """
    del_w = np.zeros(weights.shape)
    # Loop through all records, x is the input, y is the target
    for x, y in zip(features.values, targets):
        # TODO: Calculate the output of f(h) by passing h (the dot product
        # of x and weights) into the activation function (sigmoid).
        # Replace None with appropriate code
        output = sigmoid(np.dot(x, weights))

        # TODO: Calculate the error by subtracting the network output
        # from the target (y).
        # Replace None with appropriate code
        error = y - output

        # TODO: Calculate the error term by multiplying the error by the
        # gradient. Recall that the gradient of the sigmoid f(h) is
        # f(h)*(1−f(h)) so you do not need to call any additional
        # functions and can simply apply this formula to the output and
        # error you already calculated.
        # Replace None with appropriate code
        error_term = error * output * (1 - output)

        # TODO: Update the weight step by multiplying the error term by
        # the input (x) and adding this to the current weight step.
        # Replace 0 with appropriate code
        del_w += error_term * x

    n_records = features.shape[0]
    # TODO: Update the weights by adding the learning rate times the
    # change in weights divided by the number of records.
    # Replace 0 with appropriate code
    weights += learnrate * del_w / features.shape[0]

    return weights


def gradient_descent(features, targets, epochs=1000, learnrate=0.5):
    """
    Perform the complete gradient descent process on a given dataset
    """
    # Use to same seed to make debugging easier
    np.random.seed(42)

    # Initialize loss and weights
    last_loss = None
    n_features = features.shape[1]
    weights = np.random.normal(scale=1 / n_features**0.5, size=n_features)

    # Repeatedly update the weights based on the number of epochs
    for e in range(epochs):
        weights = update_weights(weights, features, targets, learnrate)

        # Printing out the MSE on the training set every 10 epochs.
        # Initially this will print the same loss every time. When all of
        # the TODOs are complete, the MSE should decrease with each
        # printout
        if e % (epochs / 10) == 0:
            out = sigmoid(np.dot(features, weights))
            loss = np.mean((out - targets) ** 2)
            if last_loss and last_loss < loss:
                print("Train loss: ", loss, "  WARNING - Loss Increasing")
            else:
                print("Train loss: ", loss)
            last_loss = loss

    return weights


# Calculate accuracy on test data
weights = gradient_descent(features, targets)
tes_out = sigmoid(np.dot(features_test, weights))
predictions = tes_out > 0.5
accuracy = np.mean(predictions == targets_test)
print("Prediction accuracy: {:.3f}".format(accuracy))